<a href="https://colab.research.google.com/github/guadalupe-santos/ALC-Laboratorios/blob/main/labo04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

def calculaLU(A):

  if A is None:
        return None, None, 0

  if (issubclass(A.dtype.type, np.integer)):
        A = A.astype(float)

  nops = 0
  m=A.shape[0]
  n=A.shape[1]
  U = A.copy()
  L = np.eye(A.shape[1])

  if m!=n:
        return None, None, 0

  for j in range(n - 1):
    for i in range(j+1,n):
      if U[j,j] != 0:
        s = U[i,j] / U[j,j]
        nops += 1
        L[i,j] = s
        fila = U[i] - s * U[j]
        U[i] = fila
        nops += (2*i) # i = n, multiplica U[j] n veces, y se lo resta n veces
      else:
        return None, None, 0

  return L, U, nops

  """for k in range(1,m):
    nops += (m-k) +2*((m-k)**2)"""

# TESTS LU
print("TESTS calculaLU")

L0 = np.array([[1,0,0],
               [0,1,0],
               [1,1,1]])

U0 = np.array([[10,1,0],
               [0,2,1],
               [0,0,1]])

A =  L0 @ U0
L,U,nops = calculaLU(A)
assert(np.allclose(L,L0))
assert(np.allclose(U,U0))


L0 = np.array([[1,0,0],
               [1,1.001,0],
               [1,1,1]])

U0 = np.array([[1,1,1],
               [0,1,1],
               [0,0,1]])
A =  L0 @ U0
L,U,nops = calculaLU(A)
assert(not np.allclose(L,L0))
assert(not np.allclose(U,U0))
assert(np.allclose(L,L0,atol=1e-3))
assert(np.allclose(U,U0,atol=1e-3))
assert(nops == 13)

L0 = np.array([[1,0,0],
               [1,1,0],
               [1,1,1]])

U0 = np.array([[1,1,1],
               [0,0,1],
               [0,0,1]])

A =  L0 @ U0
L,U,nops = calculaLU(A)
assert(L is None)
assert(U is None)
assert(nops == 0)

assert(calculaLU(None) == (None, None, 0))

assert(calculaLU(np.array([[1,2,3],[4,5,6]])) == (None, None, 0))

print("-----ÉXITO!!!!\n")



TESTS calculaLU
-----ÉXITO!!!!



Ejercicio 2

In [3]:
def res_tri(L,b,inferior= True):
  A = L.copy()
  n = L.shape[1]
  m = L.shape[0]
  x = np.zeros(m)
  if A is None:
        return None
  if (issubclass(A.dtype.type, np.integer)):
        A = A.astype(float)

  if inferior:
   for i in range(0,m):
      escalar = 0
      valor = 0
      indice = 0
      if A[i,i] != 0:
        if i == 0:
          escalar = A[i,0]
          variable = b[i] / escalar
          x[i] = variable
        else:
         for j in range(i+1):
          if A[i,j] != 0:
            escalar = A[i,j]
            valor += (A[i,j] * x[j])
      variable = (b[i] - valor)/ A[i,i]
      x[i] = variable
  else:
     for i in range(m-1,-1,-1):
      if A[i,i] != 0:
       escalar = 0
       valor = 0
       indice = 0
       if i == (m-1):
          escalar = A[i,m-1]
          variable = b[i] / escalar
          x[i] = variable
       else:
        for j in range(n-1,-1,-1):
            escalar = A[i,j]
            valor += (A[i,j] * x[j])
       variable = (b[i] - valor)/ A[i,i]
       x[i] = variable
      else:
        return None
  return x


Ejercicio 3

In [4]:
import numpy as np

""" idea:
Uy=b -> con b= x= L^-1
Lx=b -> con b= I
-> y = L^-1*(U^-1*b) = U^-1*L^-1*I = A^-1
"""
def inversa(A):

  if (issubclass(A.dtype.type, np.integer)):
        A = A.astype(float)
  if A is None:
    return None

  L, U, _ = calculaLU(A)
  m = U.shape[0]
  n = U.shape[1]
  I = np.eye(n)
  x = np.zeros((m,m))
  y = np.copy(x)

  if m!=n:
    return None

  for i in range(n):
    if abs(U[i,i]) < 1e-10:
      return None
    x[:,i] = res_tri(L,I[:,i],inferior=True)
    y[:,i] = res_tri(U,x[:,i],inferior=False)

  return y






Ejercicio 4

In [5]:
def traspuesta(A):
  T = np.copy(A)
  n = A.shape[0]
  m = A.shape[1]
  for j in range(m):
      if m == n:
        T[:,j] = A[j,:]
  return T

def calculaLDV(A):

  if (issubclass(A.dtype.type, np.integer)):
        A = A.astype(float)
  if A is None:
    return None

  L, U_, _ = calculaLU(A)
  U = traspuesta(U_)
  V_, D, _ = calculaLU(U)
  V = traspuesta(V_)
  for i in range(U.shape[0]):
    if abs(U[i,i]) < 1e-10:
      return None

  return L,D,V


Ejercicio 5

In [7]:
def esSDP(A, atol=1e-8):
  if (issubclass(A.dtype.type, np.integer)):
        A = A.astype(float)
  if A is None:
    return None

  n= A.shape[0]
  m= A.shape[1]

  if calculaLDV(A) is None:
    return False

  L, D, V = calculaLDV(A)

  for i in range(n):
    if abs(A[i,i]) > 1e-20:
      for j in range(m):
        if abs(A[i,j] - A[j,i]) < 1:
          if abs(D[i,i]) < 1e-8:
            return False
    else:
      return False
  return True


TESTS esSDP
-----ÉXITO!!!!

---FINALIZADO LABO 4!---
